In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Reading from Bronze

In [0]:
df = spark.table('workspace.bronze.crm_cust_info')

In [0]:
df.display()

Transformations
- Trim the string
- Normalisation for marital_status, gender
- column names

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name, trim(col(field.name)))
    

In [0]:
df.display()

# Normalisation

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

In [0]:
df.display()

# Remove Null CustID

In [0]:
df = df.filter(col("cst_id").isNotNull())

# Remanning Column Names

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(20).display()

# Write into Silver Table

In [0]:
(
    df
        .write.mode("overwrite")
        .format("delta")
        .saveAsTable("workspace.silver.crm_customers")
)

In [0]:
%sql
select * from workspace.silver.crm_customers limit 10